In [1]:
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import warnings
from scipy.integrate import solve_ivp
from numba import njit
from itertools import product
 
warnings.filterwarnings("ignore")
 
@njit(cache=True, fastmath=True)
def replicator_rhs(x, A):
    Ax = A@x
    avg_payoff = x@Ax
    return x*(Ax-avg_payoff)

@njit(cache=True, fastmath=True)
def jacobian_re(x, A):
    #J= diag((Ap)-pAp)+p(A-(A+A^T)p)
    Ax = A@x
    avg_payoff = x@Ax
    return np.diag(Ax-avg_payoff)+x[:, None]*(A-(A+np.transpose(A))@x)

@njit(cache=True, fastmath=True)
def lotka_volterra_rhs(t, x, A, b):
    #dot(x) = x(b+Ax)
    return x*(b+A@x)

@njit(cache=True, fastmath=True)
def jacobian_lv(x, A, b):
    # J = diag(b + A x) + diag(x) A
    return np.diag(b + A@x) + x[:, None] * A

@njit(cache=True, fastmath=True)
def rk4_eco_step(x, dt, A):
    k1 = replicator_rhs(x, A)
    k2 = replicator_rhs(x + 0.5*dt*k1, A)
    k3 = replicator_rhs(x + 0.5*dt*k2, A)
    k4 = replicator_rhs(x +     dt*k3, A)
    return x + dt*(k1 + 2*k2 + 2*k3 + k4)/6

@njit(cache=True, fastmath=True)
def rk4_variational_step(x, dt, A, k):
    k1 = variational_rhs(x, A, k)
    k2 = variational_rhs(x + 0.5*dt*k1, A, k)
    k3 = variational_rhs(x + 0.5*dt*k2, A, k)
    k4 = variational_rhs(x +     dt*k3, A, k)
    return x + dt*(k1 + 2*k2 + 2*k3 + k4)/6

@njit(cache=True, fastmath=True)
def variational_rhs(y, A, k):
    # y = [x (N,), Q (N, k) flattened]
    N = A.shape[0]
    x = y[:N]
    Q = y[N:].reshape(N, k)
    dx = replicator_rhs(x, A)
    J = jacobian_re(x, A)
    dQ = J@Q
    return np.concatenate((dx, dQ.ravel()))

#@njit(cache=True, fastmath=True)
def largest_lyapunov(A, x0, t_transient, t_total, renorm_dt, k, rng):
    """Benettin's algorithm. Returns the k largest Lyapunov exponents."""
    N = A.shape[0]
    rng = rng or np.random.default_rng()

    # 1) Burn off the transient so we are actually on the attractor.
    dt=0.01
    x = rk4_eco_step(x0, dt, A)
    for i in range (int(t_transient/dt)):
        x = rk4_eco_step(x, dt, A)

    # 2) Start with an orthonormal set of tangent vectors.
    Q, _ = np.linalg.qr(rng.standard_normal((N, k)))

    # 3) Evolve + renormalize.
    n_steps = int(t_total / renorm_dt)
    n_inner = int(renorm_dt/dt)
    log_sums = np.zeros(k)
    for _ in range(n_steps):
        y0 = np.concatenate((x, Q.ravel()))
        Y = rk4_variational_step(y0, dt, A, k)
        for i in range (n_inner):
            Y = rk4_variational_step(Y, dt, A, k)

        x = Y[:N]
        Q = Y[N:].reshape(N, k)
        Q, R = np.linalg.qr(Q)    # reorthonormalize
        log_sums += np.log(np.abs(np.diag(R)))

    return log_sums / (n_steps * renorm_dt)

def econetwork(N, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    
    a=np.ones(N)
    A=np.zeros((N,N))
    for i in range (N):
        A[i][(i+1)%(N)]=a[i]
        A[i][i]=-0.5
    return A
 
def one_job(mc_idx, k_idx, seed_mc, sigma_k, N):
    rng = np.random.default_rng(seed_mc)            # same RNG per MC across all sigmas
    A = econetwork(N, rng=rng)

    d_A = rng.random((N, N))
    mask = np.ones((N, N), dtype=bool)
    mask[np.arange(N), (np.arange(N) + 1) % N] = False
    d_A[mask] = 0

    A_fin = A + sigma_k / N * d_A

    x0 = np.linspace(1, N, N) / N
    x0 = x0 / x0.sum()
    
    t_trans = 10000
    lam_vec = largest_lyapunov(A_fin, x0,
                           t_transient=t_trans,
                           t_total=10 * t_trans,
                           renorm_dt=1, k=2, rng=rng)
    lam_0, lam_1 = lam_vec[0], lam_vec[1]
    return mc_idx, k_idx, lam_0, lam_1


 

# ------- Default parameters -------------------------------------------
N = 10                              # number of nodes
Nhet = 100                              # number of heterogeneity steps
sigma = np.linspace(0, 10, Nhet)     # heterogeneity interval
Nmc = 100      # Monte Carlo realizations

# Parallel Monte Carlo loop
base_seed = np.random.SeedSequence().entropy
mc_seeds = [base_seed + i for i in range(Nmc)]

jobs = list(product(range(Nmc), range(Nhet)))

results = Parallel(n_jobs=-1, backend="loky", batch_size="auto")(
    delayed(one_job)(i, k, mc_seeds[i], sigma[k], N) for i, k in jobs)

LyapExp_run_0 = np.empty((Nmc, Nhet))
LyapExp_run_1 = np.empty((Nmc, Nhet))
for i, k, lam_0, lam_1 in results:
    LyapExp_run_0[i, k], LyapExp_run_1[i, k] = lam_0, lam_1


# Improvement relative to sigma = 0
baseline = LyapExp_run_1[:, [0]]                         # (Nmc, 1)
stabilityimprovement = (LyapExp_run_1 - baseline)/baseline       # (Nmc, Nhet)


improved = LyapExp_run_1 < baseline
stabilizedfraction = np.sum(improved, axis=0) / Nmc

# ------- Plot ----------------------------------------------------------
Nsamps = 10
j = 0

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

ax = axes[0, 0]
ax.plot(sigma, (LyapExp_run_0[:Nsamps, :]).T,linewidth=2)
#ax.plot(sigma, np.zeros(len(sigma)), color='black')
ax.set_xlabel(r"heterogeneity $\sigma$")
ax.set_ylabel(r"$\lambda_{0}$")

ax = axes[0, 1]
ax.plot(sigma, (LyapExp_run_1[:Nsamps, :]).T,linewidth=2)
#ax.plot(sigma, np.zeros(len(sigma)), color='black')
ax.set_xlabel(r"heterogeneity $\sigma$")
ax.set_ylabel(r"$\lambda_{1}$")


#print('.')
ax = axes[1, 0]
ax.plot(sigma, np.sum(np.transpose(stabilityimprovement)), linewidth=2)
#ax.set_xlim(0, 0.5)
ax.set_xlabel(r"heterogeneity $\sigma$")
ax.set_ylabel("stability improvement")


ax = axes[1, 1]
ax.plot(sigma, stabilizedfraction, linewidth=2)
#ax.set_xlim(0, 0.5)
ax.set_xlabel(r"heterogeneity $\sigma$")
ax.set_ylabel("stabilized fraction")


for ax in axes.flat:
    for item in ([ax.title, ax.xaxis.label, ax.yaxis.label] +
                    ax.get_xticklabels() + ax.get_yticklabels()):
        item.set_fontsize(14)

plt.tight_layout()
plt.show()
 
 

KeyboardInterrupt: 